# T2.5 – EigenCAM Feature Visualisation

Visualises which regions of the input image most activate the backbone (Layer 10) and neck (Layer 22) of the best-performing (Config 3) YOLO model.

## 1. Setup & Imports

In [ ]:
# Install required libraries
!pip install -q -U ultralytics
!pip install -q grad-cam

# Imports
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from pytorch_grad_cam import EigenCAM
from pytorch_grad_cam.utils.image import show_cam_on_image


## 2. Load & Extract the Dataset

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Unzip the dataset
# (Make sure this path matches exactly where the zip file is in Drive)
!unzip -q "/content/drive/MyDrive/kitti3k_yolo.zip"

# 3. Flatten the folder structure (in case it is double-nested)
!mv /content/kitti3k_yolo/kitti3k_yolo/* /content/kitti3k_yolo/ 2>/dev/null || true

print("Dataset loaded! Ready to run EigenCAM.")


## 3. Load Model & Generate EigenCAM Heatmaps (Single Image)

In [ ]:
# --- Configuration ---
model_path = "/content/drive/MyDrive/cfg3_best.pt"
img_test = "/content/kitti3k_yolo/images/val/000136.png"

print("Loading YOLO model...")
model = YOLO(model_path)

# Extract the underlying PyTorch model to pass to EigenCAM
pytorch_model = model.model
pytorch_model.eval()


# Wrapper needed because YOLO's forward pass returns a tuple, which EigenCAM can't consume directly
class YOLOWrapper(torch.nn.Module):
    def __init__(self, pt_model):
        super().__init__()
        self.model = pt_model

    def forward(self, x):
        out = self.model(x)
        # YOLO returns a tuple; extract the main prediction tensor to satisfy EigenCAM
        if isinstance(out, tuple) or isinstance(out, list):
            return out[0]
        return out


# Wrap the model
wrapped_model = YOLOWrapper(pytorch_model)
wrapped_model.eval()

# 3. Define the target layers
# Layer 10: end of the backbone (deep semantic features)
target_layer_10 = [pytorch_model.model[10]]

# Layer 22: end of the neck (pre-detection localised features)
target_layer_22 = [pytorch_model.model[22]]


# 4. Helper function to generate and plot a heatmap
def plot_eigencam(img_path, target_layers, title):
    img = cv2.imread(img_path)
    if img is None:
        print(f"Error: Could not find image at {img_path}")
        return

    img = cv2.resize(img, (640, 640))
    rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Normalise image to float32 [0.0, 1.0]
    float_rgb_img = np.float32(rgb_img) / 255.0

    # Prepare input tensor from float_rgb_img (batch, channels, height, width)
    input_tensor = float_rgb_img.transpose((2, 0, 1))
    input_tensor = np.expand_dims(input_tensor, axis=0)

    # Convert to PyTorch Tensor on the correct device and float32 dtype
    device = next(pytorch_model.parameters()).device
    input_tensor = torch.from_numpy(input_tensor).to(device=device, dtype=torch.float32)

    # Initialise EigenCAM using the wrapped model
    cam = EigenCAM(model=wrapped_model, target_layers=target_layers)

    # Generate the grayscale heatmap
    grayscale_cam = cam(input_tensor=input_tensor)[0, :]

    # Overlay the heatmap on the original image
    cam_image = show_cam_on_image(float_rgb_img, grayscale_cam, use_rgb=True)

    # Plot the result
    plt.figure(figsize=(10, 10))
    plt.imshow(cam_image)
    plt.title(title, fontsize=16)
    plt.axis('off')

    # Save the figure locally
    safe_title = title.replace(" ", "_").replace("/", "_").replace(":", "")
    plt.savefig(f"{safe_title}.png", bbox_inches='tight', dpi=300)
    plt.show()


## 4. Generate Heatmaps for All Three Validation Images

In [ ]:
# The three validation images used for qualitative analysis in Task 2.4
image_ids = ['000136', '000232', '000189']

for img_id in image_ids:
    img_path = f"/content/kitti3k_yolo/images/val/{img_id}.png"

    print(f"\n{'='*40}")
    print(f"=== Generating Heatmaps for Image: {img_id} ===")
    print(f"{'='*40}")

    # Generate Layer 10 (backbone)
    plot_eigencam(img_path, target_layer_10, f"Layer 10 Backbone - Image {img_id}")

    # Generate Layer 22 (neck)
    plot_eigencam(img_path, target_layer_22, f"Layer 22 Neck - Image {img_id}")
